# 05: ディメンショナルモデリング（Gold 層）と WAP パターン

## このノートブックで学習できること

- Bronze / Silver 層のデータから、Gold 層（`gold` 名前空間）にスタースキーマを構成する 5 テーブル（`fct_sales`、`dim_customers` SCD Type 2、`dim_products`、`dim_stores`、`dim_date`）を構築する手順
- DuckDB を用いて Gold 層のスタースキーマを JOIN し、地域別・会員ランク別・商品カテゴリ別の売上集計を行う分析クエリ
- Day 3 の異常データ（`total_amount` のマイナス値・NULL が 20% 混入）を取り込む際、本番（`main`）を汚さずに隔離ブランチ（`audit_branch`）へ書き込み（Write）、品質監査（Audit）で異常を検知して本番反映（Publish）を中止・破棄する WAP（Write-Audit-Publish）パターンの実践
- 月末決算確定スナップショットへの `TAG` 付与（`closing_2026_11`）

## 到達ゴール

- スタースキーマのファクト・ディメンション設計と SCD Type 2 の結合キー（サロゲートキー）の役割を実装を通じて理解する
- 不正データが本番テーブルへ流出するのをブランチ隔離と品質検査で防ぐ WAP フローを実装できる

## なぜこれが必要か

分析用の Gold テーブルに異常値や欠損データが一時的にでも書き込まれると、BI ダッシュボードや経営レポートが誤った数値を表示してしまいます。WAP パターンを使うことで、検査に合格したデータだけをアトミックに本番へ公開できます。


## 1. Gold 層スタースキーマ（5 テーブル）の構築

このセルでは、Day 1〜Day 2 の正常なデータ（Bronze / Silver / Landing のマスタ）をもとに、Gold 層に `dim_customers`（SCD Type 2）、`dim_products`、`dim_stores`、`dim_date`、`fct_sales` の 5 つの Iceberg テーブルを作成します。


In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.catalog_helper import (
    LANDING_DIR,
    LAKEHOUSE_DIR,
    get_duckdb_conn,
    get_iceberg_catalog,
    init_storage,
    register_iceberg_table,
)
from datetime import date, timedelta
import polars as pl
import pyarrow as pa

catalog = get_iceberg_catalog()
for t in ["gold.dim_customers", "gold.dim_products", "gold.dim_stores", "gold.dim_date", "gold.fct_sales"]:
    if catalog.table_exists(t):
        catalog.drop_table(t)

# 1) dim_customers (Notebook 02 の silver.crm_customers_scd2 を元にサロゲートキー付与)
scd2_df = pl.from_arrow(catalog.load_table("silver.crm_customers_scd2").scan().to_arrow())
dim_customers = scd2_df.sort(["customer_id", "valid_from"]).with_row_index("customer_key", offset=1).with_columns([
    pl.lit("***MASKED***").alias("email_masked")
]).select(["customer_key", "customer_id", "email_masked", "membership_tier", "status", "valid_from", "valid_to", "is_current"])

# 2) dim_products (Notebook 03 の bronze.erp_products を元にする)
prod_df = pl.from_arrow(catalog.load_table("bronze.erp_products").scan().to_arrow())
dim_products = prod_df.sort("product_id").with_row_index("product_key", offset=1).select([
    "product_key", "product_id", "product_name", "category", "unit_price", "cost"
])

# 3) dim_stores
store_df = pl.read_csv(LANDING_DIR / "day2" / "store_master.csv")
dim_stores = store_df.sort("store_id").with_row_index("store_key", offset=1).select([
    "store_key", "store_id", "store_name", "region", "prefecture", "opened_date"
])

# 4) dim_date (2026-11-01 .. 2026-11-30)
dates = [date(2026, 11, 1) + timedelta(days=i) for i in range(30)]
dim_date = pl.DataFrame([
    {
        "date_key": int(d.strftime("%Y%m%d")),
        "date": d.isoformat(),
        "day_of_week": d.strftime("%a"),
        "is_weekend": d.weekday() >= 5,
        "month": d.month,
        "quarter": (d.month - 1) // 3 + 1,
        "year": d.year,
    }
    for d in dates
])

# 5) fct_sales (Notebook 03 の bronze.pos_transactions のうち quantity 正常行)
pos_df = pl.from_arrow(catalog.load_table("bronze.pos_transactions").scan().to_arrow()).filter(pl.col("quantity").is_not_null())
pos_enriched = (
    pos_df.with_columns([
        pl.col("purchased_at").str.slice(0, 10).str.replace_all("-", "").cast(pl.Int64).alias("date_key"),
        pl.col("discount_amount").fill_null(0).alias("discount_amount"),
        pl.col("payment_method").fill_null("UNKNOWN").alias("payment_method"),
    ])
    .join(dim_customers.filter(pl.col("is_current")).select(["customer_id", "customer_key"]), on="customer_id", how="left")
    .join(dim_products.select(["product_id", "product_key"]), on="product_id", how="left")
    .join(dim_stores.select(["store_id", "store_key"]), on="store_id", how="left")
    .with_columns([
        (pl.col("quantity") * pl.col("unit_price")).alias("gross_amount"),
        pl.col("total_amount").alias("net_amount"),
    ])
    .select([
        pl.col("transaction_id").alias("sale_id"),
        "date_key", "customer_key", "product_key", "store_key",
        "payment_method", "quantity", "gross_amount", "discount_amount", "net_amount", "purchased_at",
    ])
)

gold_tables = {}
for name, df in [
    ("gold.dim_customers", dim_customers),
    ("gold.dim_products", dim_products),
    ("gold.dim_stores", dim_stores),
    ("gold.dim_date", dim_date),
    ("gold.fct_sales", pos_enriched),
]:
    tbl = catalog.create_table(name, schema=df.to_arrow().schema)
    tbl.append(df.to_arrow())
    gold_tables[name] = tbl
    print(f"{name}: {df.height} 行")


Gold 層の 5 テーブルが構築されました。

次のセルでは、DuckDB にこれら 5 つの Iceberg テーブルを登録し、地域別 × 会員ランク別 × 商品カテゴリ別の売上分析クエリを実行します。


In [ ]:
con = get_duckdb_conn()
for short_name in ["fct_sales", "dim_customers", "dim_products", "dim_stores", "dim_date"]:
    register_iceberg_table(con, gold_tables[f"gold.{short_name}"], short_name)

con.sql("""
SELECT
    s.region,
    c.membership_tier,
    p.category,
    COUNT(*) AS tx_count,
    SUM(f.net_amount) AS total_net_sales
FROM fct_sales f
JOIN dim_stores s ON f.store_key = s.store_key
JOIN dim_customers c ON f.customer_key = c.customer_key
JOIN dim_products p ON f.product_key = p.product_key
GROUP BY 1, 2, 3
ORDER BY total_net_sales DESC
LIMIT 10
""").pl()


スタースキーマの JOIN により、地域・会員ランク・商品カテゴリ別の売上集計が高速に実行できました。

## 2. WAP（Write-Audit-Publish）パターンによる Day 3 異常データの遮断とタグ付与

次のセルでは、現在の健全な `gold.fct_sales` に月次確定タグ `closing_2026_11` を付けたうえで、Day 3 の `pos_transactions.jsonl`（約 20% に `total_amount < 0` または `NULL` を含む）を隔離ブランチ `audit_branch` に書き込み（Write）ます。その後、DuckDB で品質監査（Audit）を実行し、異常が検知されたため `main` ブランチへマージせずに処理を中断して `main` が無傷であることを確認します。


In [ ]:
import json

fct_tbl = gold_tables["gold.fct_sales"]
main_snap_before = fct_tbl.current_snapshot().snapshot_id
fct_tbl.manage_snapshots().create_tag(main_snap_before, "closing_2026_11").commit()

# Day 3 データを隔離ブランチ audit_branch へ Write
d3_raw = [json.loads(l) for l in (LANDING_DIR / "day3" / "pos_transactions.jsonl").read_text(encoding="utf-8").splitlines()]
d3_df = (
    pl.DataFrame(d3_raw)
    .with_columns([
        pl.col("purchased_at").str.slice(0, 10).str.replace_all("-", "").cast(pl.Int64).alias("date_key"),
    ])
    .join(dim_customers.filter(pl.col("is_current")).select(["customer_id", "customer_key"]), on="customer_id", how="left")
    .join(dim_products.select(["product_id", "product_key"]), on="product_id", how="left")
    .join(dim_stores.select(["store_id", "store_key"]), on="store_id", how="left")
    .with_columns([
        (pl.col("quantity") * pl.col("unit_price")).alias("gross_amount"),
        pl.col("total_amount").alias("net_amount"),
    ])
    .select([
        pl.col("transaction_id").alias("sale_id"),
        "date_key", "customer_key", "product_key", "store_key",
        "payment_method", "quantity", "gross_amount", "discount_amount", "net_amount", "purchased_at",
    ])
)

fct_tbl.manage_snapshots().create_branch(main_snap_before, "audit_branch").commit()
# audit_branch へ追記
fct_tbl.append(d3_df.to_arrow(), branch="audit_branch")
audit_snap = fct_tbl.snapshot_by_name("audit_branch").snapshot_id

# Audit: DuckDB で audit_branch を検査
register_iceberg_table(con, fct_tbl, "fct_audit", snapshot_id=audit_snap)
audit_res = con.execute("""
SELECT
    COUNT(*) FILTER (WHERE net_amount < 0) AS negative_cnt,
    COUNT(*) FILTER (WHERE net_amount IS NULL) AS null_cnt
FROM fct_audit
""").fetchone()

print(f"Audit 結果 -> マイナス売上: {audit_res[0]} 件 / NULL: {audit_res[1]} 件")
if audit_res[0] > 0 or audit_res[1] > 0:
    print("品質検査不合格: main への Publish を中止し、本番テーブルは無傷に保たれました。")

print(f"本番 main の行数: {fct_tbl.scan().to_arrow().num_rows} 行")
print(f"タグ closing_2026_11 の行数: {fct_tbl.scan(snapshot_id=fct_tbl.snapshot_by_name('closing_2026_11').snapshot_id).to_arrow().num_rows} 行")


`audit_branch` には異常値（マイナス 30 件・NULL 30 件）が検出されたため、本番 `main` ブランチへの公開は遮断され、`main` と `closing_2026_11` タグはともに健全な 590 行のまま保護されました。
